In [ ]:
# Gold 통합 노트북: 4h 가격 × 공포탐욕 — 대시보드 통합 뷰
# - 실행 순서: gold/fear_greed_metrics.ipynb를 먼저 실행한 뒤 이 노트북을 실행하세요.
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ===== (A) 공통 설정 =====
CATALOG = "demo_catalog"
SCHEMA  = "demo_schema"
PRICES  = f"{CATALOG}.{SCHEMA}.gold_prices_4h"          # 4h 가격 + MA + GC/DC
FNG_GLD = f"{CATALOG}.{SCHEMA}.gold_fear_greed"         # 일단위 Fear & Greed (단일 Gold)
GOLD_J  = f"{CATALOG}.{SCHEMA}.gold_price_positions_4h" # 2원 조인 집계 테이블
DAYS_BACK = 120           # 최근 N일만 재빌드
FNG_FRESH_DAYS = 3        # FNG 신선도 제한(일). 초과 시 null 처리

spark.sql("SET spark.sql.session.timeZone=UTC")

# ===== Spark / Delta Performance Configuration =====
# optimizeWrite: 커밋 전 소파일을 병합 → 소파일 누적 방지
spark.conf.set("spark.databricks.delta.optimizeWrite", "true")
# autoCompact: 쓰기 완료 후 백그라운드 컴팩션 자동 트리거
spark.conf.set("spark.databricks.delta.autoCompact", "true")
# AQE: broadcast join 및 dedup window에서 런타임 통계 기반 동적 재플래닝
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# ===== (A-1) 선행 테이블 검증 =====
# joined_dashboard는 fear_greed_metrics의 산출물을 그대로 소비한다.
# 선행 노트북이 실행되지 않은 상태에서 조용히 빈 결과를 만드는 대신 즉시 실패시킨다.
if not spark.catalog.tableExists(FNG_GLD):
    raise RuntimeError(
        f"[선행 조건 미충족] {FNG_GLD} 테이블이 없습니다. "
        f"gold/fear_greed_metrics.ipynb 를 먼저 실행하세요."
    )

# ===== (B) 4h 가격 버킷 =====
prices = (
    spark.table(PRICES)
         .where(f"dt >= date_sub(current_date(), {DAYS_BACK})")
         .select(
             "symbol", "bucket_start", "close_4h", "ma50_4h", "ma200_4h",
             "cross_signal", "pct_change_24h", "dt"
         )
         .withColumn("bucket_end", F.col("bucket_start") + F.expr("INTERVAL 4 HOURS"))
)

# ===== (C) Fear & Greed as-of 매핑 (bucket_end 이전 최신 1건) =====
# FNG는 연 최대 ~365행의 소형 테이블이므로 broadcast()로 map-side join 강제
# → 대용량 prices 테이블과의 셔플(Network I/O)을 완전히 제거
# AQE 런타임 브로드캐스트 전환(방어 레이어)과 함께 사용하여 이중 안전장치 확보
fng = (
    spark.table(FNG_GLD)
         .select(
             F.col("ts_utc").alias("fng_ts"),
             F.col("value").cast("int").alias("fng_value"),
             F.col("value_class").alias("fng_label")
         )
)

gold_df = (
    prices.alias("p")
      .join(F.broadcast(fng).alias("f"), F.col("f.fng_ts") <= F.col("p.bucket_end"), "left")
      .withColumn(
          "rn",
          F.row_number().over(
              Window.partitionBy("p.bucket_start").orderBy(F.col("f.fng_ts").desc())
          )
      )
      .where("rn = 1")
      # 신선도 제한: 오래되면 null
      .withColumn(
          "fng_value",
          F.when(
              F.datediff(F.col("p.bucket_end"), F.col("f.fng_ts")) <= FNG_FRESH_DAYS,
              F.col("f.fng_value")
          ).otherwise(F.lit(None).cast("int"))
      )
      .withColumn(
          "fng_label",
          F.when(
              F.datediff(F.col("p.bucket_end"), F.col("f.fng_ts")) <= FNG_FRESH_DAYS,
              F.col("f.fng_label")
          ).otherwise(F.lit(None).cast("string"))
      )
      .selectExpr(
          "p.symbol as symbol", "p.bucket_start as bucket_start", "p.bucket_end as bucket_end", "p.dt as dt",
          "p.close_4h as close_4h", "p.ma50_4h as ma50_4h", "p.ma200_4h as ma200_4h",
          "p.cross_signal as cross_signal", "p.pct_change_24h as pct_change_24h",
          "fng_value", "fng_label"
      )
)

# ===== (D) 집계 테이블 생성 및 MERGE (idempotent upsert) =====
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_J} (
  symbol          STRING,
  bucket_start    TIMESTAMP,
  bucket_end      TIMESTAMP,
  close_4h        DOUBLE,
  ma50_4h         DOUBLE,
  ma200_4h        DOUBLE,
  cross_signal    STRING,
  pct_change_24h  DOUBLE,
  fng_value       INT,
  fng_label       STRING,
  dt              DATE
) USING DELTA
PARTITIONED BY (dt)
TBLPROPERTIES (
  'delta.logRetentionDuration'         = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 14 days'
)
""")

target_j = DeltaTable.forName(spark, GOLD_J)
(target_j.alias("t")
  .merge(
      gold_df.alias("s"),
      "t.symbol = s.symbol AND t.bucket_start = s.bucket_start AND t.dt = s.dt"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)
print(f"[JOIN GOLD] MERGE complete: {GOLD_J}")
